# Composition vs. Inheritance — Example

We model vehicles that can move in different ways. First we try **inheritance**, then we solve the same problem with **composition** and compare.

## Attempt 1: Inheritance

We start with a `Vehicle` base class and add a `Car` and a `Boat`. So far so good — both truly *are* vehicles.

In [ ]:
class Vehicle:
    def move(self) -> str:
        raise NotImplementedError


class Car(Vehicle):
    def move(self) -> str:
        return "driving on the road"


class Boat(Vehicle):
    def move(self) -> str:
        return "sailing on water"


for v in (Car(), Boat()):
    print(v.move())

## Where inheritance breaks down

Now product wants an **amphibious car** that can both drive and sail, and later a **flying car**. With single inheritance we're stuck picking one parent, or we start multiplying subclasses for every combination:

```text
Car, Boat, FlyingVehicle, AmphibiousCar(Car, Boat)?, FlyingCar(Car, FlyingVehicle)?, FlyingAmphibiousCar(...)?
```

This is the classic combinatorial explosion of behaviors that inheritance-only designs run into.

In [ ]:
class FlyingVehicle(Vehicle):
    def move(self) -> str:
        return "flying through the air"


# To get an amphibious car we'd need multiple inheritance, and every new
# movement style doubles the number of subclasses needed to cover combinations.
class AmphibiousCar(Car, FlyingVehicle):
    def move(self) -> str:
        # Which parent's behavior wins? We have to hand-write this anyway.
        return f"{Car.move(self)} AND {FlyingVehicle.move(self)}"


print(AmphibiousCar().move())

## Attempt 2: Composition

Instead of inheriting a fixed movement style, each `Vehicle` **has** a list of movement-behavior objects it delegates to. Adding a new combination just means composing existing behaviors — no new subclass required.

In [ ]:
from typing import Protocol


class MovementBehavior(Protocol):
    def move(self) -> str: ...


class Drive:
    def move(self) -> str:
        return "driving on the road"


class Sail:
    def move(self) -> str:
        return "sailing on water"


class Fly:
    def move(self) -> str:
        return "flying through the air"

In [ ]:
class ComposedVehicle:
    """Has-a list of movement behaviors instead of is-a fixed vehicle type."""

    def __init__(self, name: str, behaviors: list[MovementBehavior]):
        self.name = name
        self._behaviors = behaviors

    def move(self) -> str:
        actions = " AND ".join(b.move() for b in self._behaviors)
        return f"{self.name} is {actions}"


car = ComposedVehicle("Car", [Drive()])
amphibious_car = ComposedVehicle("Amphibious car", [Drive(), Sail()])
flying_amphibious_car = ComposedVehicle("Flying amphibious car", [Drive(), Sail(), Fly()])

for v in (car, amphibious_car, flying_amphibious_car):
    print(v.move())

## Reflection

- With inheritance, every new combination of behaviors risked a new subclass and manual conflict resolution (`Car.move(self)` vs `FlyingVehicle.move(self)`).
- With composition, new combinations are just new lists of behavior objects — no new classes, and behaviors (`Drive`, `Sail`, `Fly`) can even be swapped at runtime.
- Composition here still uses a small shared `MovementBehavior` interface (a `Protocol`) — composition doesn't reject polymorphism, it just applies it to the small delegated part instead of the whole class.

See `comparison-report.md` for guidance on when the inheritance version would actually have been the right call.